In [0]:
activities = spark.read.parquet("abfss://german-election-data@germanelactionstorage.dfs.core.windows.net/silver/bundestag/activities/")
descriptors = spark.read.parquet("abfss://german-election-data@germanelactionstorage.dfs.core.windows.net/silver/bundestag/activity_descriptors/")
procedures = spark.read.parquet("abfss://german-election-data@germanelactionstorage.dfs.core.windows.net/silver/bundestag/activity_procedures/")

In [0]:
procedures.printSchema()
print(procedures.columns)

root
 |-- activity_id: string (nullable = true)
 |-- procedure_id: string (nullable = true)
 |-- procedure_title: string (nullable = true)
 |-- procedure_position: string (nullable = true)
 |-- procedure_type: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)

['activity_id', 'procedure_id', 'procedure_title', 'procedure_position', 'procedure_type', 'year', 'month']


In [0]:
activities.printSchema()
print(activities.columns)

root
 |-- id: string (nullable = true)
 |-- person_id: string (nullable = true)
 |-- wahlperiode: long (nullable = true)
 |-- datum: date (nullable = true)
 |-- aktualisiert: timestamp (nullable = true)
 |-- aktivitaetsart: string (nullable = true)
 |-- dokumentart: string (nullable = true)
 |-- titel: string (nullable = true)
 |-- fundstelle_id: string (nullable = true)
 |-- dokumentnummer: string (nullable = true)
 |-- drucksachetyp: string (nullable = true)
 |-- herausgeber: string (nullable = true)
 |-- fundstelle_datum: date (nullable = true)
 |-- verteildatum: date (nullable = true)
 |-- pdf_url: string (nullable = true)
 |-- xml_url: string (nullable = true)
 |-- anfangsseite: long (nullable = true)
 |-- endseite: long (nullable = true)
 |-- urheber: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)

['id', 'person_id', 'wahlperiode', 'datum', 'aktualisiert', 'aktivitaetsart', 'dokum

In [0]:
descriptors.printSchema()
print(descriptors.columns)


root
 |-- activity_id: string (nullable = true)
 |-- descriptor_name: string (nullable = true)
 |-- descriptor_type: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)

['activity_id', 'descriptor_name', 'descriptor_type', 'year', 'month']


In [0]:
print("ACTIVITIES:", activities.count())
print("DESCRIPTORS:", descriptors.count())
print("PROCEDURES:", procedures.count())

print(
    "Activities duplicates:",
    activities.count() - activities.distinct().count()
)

print(
    "Descriptors duplicates:",
    descriptors.count() - descriptors.distinct().count()
)

print(
    "Procedures duplicates:",
    procedures.count() - procedures.distinct().count()
)

ACTIVITIES: 483988
DESCRIPTORS: 201642
PROCEDURES: 521574
Activities duplicates: 0
Descriptors duplicates: 0
Procedures duplicates: 0


In [0]:
from pyspark.sql import functions as F
procedures.groupBy("activity_id","procedure_id").count().filter(
    F.col("count") > 1
).show(30, truncate=False)

+-----------+------------+-----+
|activity_id|procedure_id|count|
+-----------+------------+-----+
|1290107    |243245      |2    |
|1290061    |245647      |2    |
|1289922    |243003      |2    |
|1289922    |244875      |2    |
|1289915    |243002      |2    |
|1289684    |244281      |2    |
|1289241    |246221      |2    |
|1289222    |244441      |2    |
|1284181    |242045      |2    |
|1284152    |245667      |2    |
|1284135    |246157      |2    |
|1284078    |234595      |2    |
|1283738    |243006      |2    |
|1283737    |243006      |2    |
|1283572    |245710      |2    |
|1283562    |243800      |2    |
|1283559    |245785      |2    |
|1288289    |240206      |2    |
|1288288    |240206      |2    |
|1288277    |240206      |2    |
|1288242    |246342      |2    |
|1288240    |246360      |2    |
|1288216    |245694      |2    |
|1288211    |246343      |2    |
|1287980    |246521      |2    |
|1287979    |246521      |2    |
|1301306    |248455      |2    |
|1301260  

In [0]:
procedures.filter(
    F.col("activity_id") == "1290107"
).orderBy(
    "year",
    "month"
).show(truncate=False)

+-----------+------------+------------------------------------------------------------------------------------------------------------+------------------+----------------------------------------+----+-----+
|activity_id|procedure_id|procedure_title                                                                                             |procedure_position|procedure_type                          |year|month|
+-----------+------------+------------------------------------------------------------------------------------------------------------+------------------+----------------------------------------+----+-----+
|1290107    |243245      |Zwölfte Verordnung zur Änderung der Außenwirtschaftsverordnung - aufhebbare Verordnung                      |Beratung          |Rechtsverordnung (Außenwirtschaftsrecht)|2019|4    |
|1290107    |239085      |Attraktivität Deutschlands für ausländisches Kapital sichern                                                |Beratung          |Antrag            

In [0]:
activities.groupBy("id").agg(
    F.count("*").alias("rows"),
    F.collect_set("year").alias("years"),
    F.collect_set("month").alias("months"),
    F.collect_set("aktualisiert").alias("updates")
).filter(
    F.col("rows") > 1
).show(200, truncate=False)

+-------+----+------+------+---------------------+
|id     |rows|years |months|updates              |
+-------+----+------+------+---------------------+
|1288014|2   |[2019]|[4, 5]|[2022-07-26 17:57:10]|
|1282586|2   |[2019]|[5, 4]|[2022-07-26 17:57:15]|
|1285659|2   |[2019]|[5, 4]|[2025-07-28 07:37:54]|
|1283359|2   |[2019]|[4, 5]|[2022-07-26 17:57:15]|
|1281036|2   |[2019]|[5, 4]|[2022-07-26 17:57:10]|
|1281019|2   |[2019]|[5, 4]|[2023-07-31 10:42:30]|
|1281621|2   |[2019]|[4, 5]|[2022-07-26 17:57:15]|
|1284450|2   |[2019]|[4, 5]|[2022-07-26 17:57:15]|
|1299494|2   |[2019]|[4, 5]|[2022-07-26 17:57:10]|
|1299179|2   |[2019]|[4, 5]|[2025-03-11 09:52:04]|
|1281189|2   |[2019]|[4, 5]|[2025-03-11 15:47:18]|
|1281146|2   |[2019]|[5, 4]|[2025-03-11 10:21:47]|
|1302230|2   |[2019]|[4, 5]|[2025-09-10 07:24:42]|
|1302108|2   |[2019]|[5, 4]|[2026-05-07 10:18:05]|
|1302601|2   |[2019]|[4, 5]|[2022-07-26 17:57:10]|
|1292423|2   |[2019]|[4, 5]|[2025-01-14 10:29:20]|
|1289537|2   |[2019]|[5, 4]|[20

In [0]:
from pyspark.sql import functions as F

compare_cols = [
    c for c in activities.columns
    if c not in ["month"]
]

activities_check = (
    activities
    .withColumn(
        "row_hash",
        F.sha2(
            F.concat_ws(
                "||",
                *[
                    F.coalesce(F.col(c).cast("string"), F.lit("NULL"))
                    for c in compare_cols
                ]
            ),
            256
        )
    )
)

In [0]:
activities_check.groupBy("id").agg(
    F.count("*").alias("rows"),
    F.countDistinct("row_hash").alias("different_versions"),
    F.collect_set("month").alias("months")
).filter(
    F.col("rows") > 1
).show(100, truncate=False)

+-------+----+------------------+------+
|id     |rows|different_versions|months|
+-------+----+------------------+------+
|1288014|2   |1                 |[5, 4]|
|1282586|2   |1                 |[4, 5]|
|1285659|2   |1                 |[4, 5]|
|1283359|2   |1                 |[4, 5]|
|1281036|2   |1                 |[5, 4]|
|1281019|2   |1                 |[5, 4]|
|1281621|2   |1                 |[5, 4]|
|1284450|2   |1                 |[4, 5]|
|1299494|2   |1                 |[4, 5]|
|1299179|2   |1                 |[4, 5]|
|1281189|2   |1                 |[4, 5]|
|1281146|2   |1                 |[4, 5]|
|1302230|2   |1                 |[5, 4]|
|1302108|2   |1                 |[4, 5]|
|1302601|2   |1                 |[4, 5]|
|1292423|2   |1                 |[5, 4]|
|1289537|2   |1                 |[5, 4]|
|1279420|2   |1                 |[4, 5]|
|1279378|2   |1                 |[5, 4]|
|1289898|2   |1                 |[4, 5]|
|1289214|2   |1                 |[4, 5]|
|1288249|2   |1 

In [0]:
gold_bundestag_activities = (
    activities
    .dropDuplicates(["id"])
    .withColumn("year", F.year("datum"))
    .withColumn("month", F.month("datum"))
    .withColumnRenamed("id", "activity_id")
)

In [0]:
procedures_clean = procedures.filter(
    ~(
        (F.col("year") == 2019) &
        (F.col("month") == 5)
    )
)

descriptors_clean = descriptors.filter(
    ~(
        (F.col("year") == 2019) &
        (F.col("month") == 5)
    )
)

In [0]:
activity_time = gold_bundestag_activities.select(
    "activity_id",
    "year",
    "month"
)

In [0]:
gold_bundestag_procedures = (
    procedures_clean
    .drop("year", "month")
    .join(
        activity_time,
        on="activity_id",
        how="left"
    )
)

gold_bundestag_descriptors = (
    descriptors_clean
    .drop("year", "month")
    .join(
        activity_time,
        on="activity_id",
        how="left"
    )
)

In [0]:
print(
    "Procedures exact duplicates:",
    gold_bundestag_procedures.count()
    - gold_bundestag_procedures.distinct().count()
)

print(
    "Descriptors exact duplicates:",
    gold_bundestag_descriptors.count()
    - gold_bundestag_descriptors.distinct().count()
)

Procedures exact duplicates: 21978
Descriptors exact duplicates: 8723


In [0]:
gold_bundestag_procedures.filter(
    F.col("year").isNull() | F.col("month").isNull()
).count()

gold_bundestag_descriptors.filter(
    F.col("year").isNull() | F.col("month").isNull()
).count()

0

In [0]:
gold_bundestag_procedures = (
    gold_bundestag_procedures
    .dropDuplicates()
)

gold_bundestag_descriptors = (
    gold_bundestag_descriptors
    .dropDuplicates()
)

In [0]:
print(
    "Procedures duplicates:",
    gold_bundestag_procedures.count()
    - gold_bundestag_procedures.distinct().count()
)

print(
    "Descriptors duplicates:",
    gold_bundestag_descriptors.count()
    - gold_bundestag_descriptors.distinct().count()
)

Procedures duplicates: 0
Descriptors duplicates: 0


In [0]:
gold_bundestag_procedures.groupBy(
    "activity_id",
    "procedure_id",
    "procedure_position"
).count().filter(
    F.col("count") > 1
).show(50, truncate=False)

+-----------+------------+------------------+-----+
|activity_id|procedure_id|procedure_position|count|
+-----------+------------+------------------+-----+
+-----------+------------+------------------+-----+



In [0]:
gold_bundestag_descriptors.groupBy(
    "activity_id",
    "descriptor_name",
    "descriptor_type"
).count().filter(
    F.col("count") > 1
).show(50, truncate=False)

+-----------+---------------+---------------+-----+
|activity_id|descriptor_name|descriptor_type|count|
+-----------+---------------+---------------+-----+
+-----------+---------------+---------------+-----+



In [0]:
gold_activities_path = (
    "abfss://german-election-data@germanelactionstorage.dfs.core.windows.net/"
    "gold/bundestag/activities/"
)

gold_descriptors_path = (
    "abfss://german-election-data@germanelactionstorage.dfs.core.windows.net/"
    "gold/bundestag/activity_descriptors/"
)

gold_procedures_path = (
    "abfss://german-election-data@germanelactionstorage.dfs.core.windows.net/"
    "gold/bundestag/activity_procedures/"
)

(
    gold_bundestag_activities
    .write
    .mode("overwrite")
    .partitionBy("year", "month")
    .parquet(gold_activities_path)
)

(
    gold_bundestag_descriptors
    .write
    .mode("overwrite")
    .partitionBy("year", "month")
    .parquet(gold_descriptors_path)
)

(
    gold_bundestag_procedures
    .write
    .mode("overwrite")
    .partitionBy("year", "month")
    .parquet(gold_procedures_path)
)

In [0]:
gold_bundestag_summary = (
    gold_bundestag_activities
    .groupBy(
        "year",
        "month",
        "aktivitaetsart",
        "dokumentart"
    )
    .agg(
        F.countDistinct("activity_id").alias("anzahl_aktivitaeten")
    )
)

In [0]:
gold_bundestag_summary_path = (
    "abfss://german-election-data@germanelactionstorage.dfs.core.windows.net/"
    "gold/bundestag/activity_summary/"
)
(
    gold_bundestag_summary
    .write
    .mode("overwrite")
    .partitionBy("year")
    .parquet(gold_bundestag_summary_path)
)